# 🚀 Bailout — Production Scheduling Assistant

> *Turn historical production data into real-time decisions*

### Prerequisites
Before running, make sure you have the following secrets set in **Colab → Secrets** (🔑 icon on the left sidebar):
- `NGROK_API_KEY` — from [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken)
- `OPENAI_API_KEY` — from [OpenAI platform](https://platform.openai.com/api-keys)

---
### Steps
Run each cell in order. The last cell will print a public URL — open it to access the Control Panel.

## 1. Clone repository

In [ ]:
!git clone https://github.com/ThuyHaLE/Bailout.git

In [ ]:
%cd Bailout

## 2. Configure API keys

In [ ]:
from google.colab import userdata

# Write .env from Colab secrets
openai_key = userdata.get('OPENAI_API_KEY')

with open('.env', 'w') as f:
    f.write(f'OPENAI_API_KEY={openai_key}\n')

print('✓ .env written')

## 3. Install Python dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q pyngrok
print('✓ Dependencies installed')

## 4. Build frontend

In [ ]:
%%bash
cd control_panel
npm install --silent
npm run build
echo '✓ Frontend built'

## 5. Launch server

In [ ]:
from google.colab import userdata
from pyngrok import ngrok
import nest_asyncio
import uvicorn

# Auth ngrok
ngrok.kill()
ngrok.set_auth_token(userdata.get('NGROK_API_KEY'))

# Patch asyncio for Colab
nest_asyncio.apply()

# Import app after deps are installed
from main import app

# Open tunnel
public_url = ngrok.connect(8000)
print('─' * 50)
print(f'🟢 Control Panel: {public_url}')
print('─' * 50)
print('Keep this cell running. Stop it to shut down.')

# Start server
config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning')
server = uvicorn.Server(config)
await server.serve()